<a href="https://colab.research.google.com/github/vaibhav1163638/Student-Pass-Fail-Prediction/blob/main/Student-Pass-Fail-Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student-Pass-Fail-Prediction

In this project, we will analyze a student exam performance dataset and build a machine learning model to predict whether a student will pass or fail.

##1. Load the Dataset

In [62]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score\

path = kagglehub.dataset_download("mobeenfatimah/student-exam-performance-and-success-dataset")
print("Dataset path:", path)
df = pd.read_csv(path + "/student_exam_performance.csv")

Using Colab cache for faster access to the 'student-exam-performance-and-success-dataset' dataset.
Dataset path: /kaggle/input/student-exam-performance-and-success-dataset


## 2. Preview the Dataset

In [63]:
df.head()

,student_id,age,gender,education_level,school_type,family_income,parent_education,urban_rural,previous_exam_score,previous_gpa,...,exam_difficulty,exam_preparation_days,questions_attempted,questions_correct,time_management_score,exam_anxiety_level,exam_score,performance_grade,pass_status,performance_level
0,STU_000001,20,Female,High School,Public,Middle,NaN,Rural,78.36,2.95,...,Medium,27,95,86,NaN,9.39,90.40,A,Pass,High
1,STU_000002,17,Male,High School,Public,Middle,NaN,Suburban,73.40,2.89,...,Easy,4,100,86,87.17,6.31,86.21,A,Pass,High
2,STU_000003,18,Other,Undergraduate,Public,Middle,Master,Urban,82.76,3.35,...,Easy,28,96,74,63.77,9.87,76.11,B,Pass,Medium
3,STU_000004,20,Male,High School,Public,Middle,High School,Suburban,60.61,2.57,...,Medium,16,97,77,59.51,9.68,79.74,B,Pass,Medium
4,STU_000005,16,Female,High School,Private,High,Bachelor,Suburban,74.79,2.84,...,Medium,28,98,52,84.64,10.00,50.66,D,Pass,Low


## 3. Understand the Dataset Structure

In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 44 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   student_id                  100000 non-null  object 
 1   age                         100000 non-null  int64  
 2   gender                      100000 non-null  object 
 3   education_level             100000 non-null  object 
 4   school_type                 100000 non-null  object 
 5   family_income               100000 non-null  object 
 6   parent_education            93474 non-null   object 
 7   urban_rural                 100000 non-null  object 
 8   previous_exam_score         100000 non-null  float64
 9   previous_gpa                92207 non-null   float64
 10  attendance_percentage       90137 non-null   float64
 11  assignment_completion_rate  100000 non-null  float64
 12  class_participation         100000 non-null  object 
 13  study_hours_per

## 4. Select Relevant Features

In [65]:
df2=df.iloc[:,[1,2,8,9,10,11,13,14,21,23,26,30,35,40,42]]
df2

,age,gender,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,sleep_hours,physical_activity_hours,internet_access,exam_preparation_days,exam_score,pass_status
0,20,Female,78.36,2.95,81.49,65.18,4.94,2.63,8,6.31,0.93,1,27,90.40,Pass
1,17,Male,73.40,2.89,NaN,76.53,3.42,2.64,10,6.81,0.21,1,4,86.21,Pass
2,18,Other,82.76,3.35,100.00,70.39,1.06,0.83,5,6.53,2.56,1,28,76.11,Pass
3,20,Male,60.61,2.57,84.91,75.06,7.89,5.51,8,6.60,2.30,1,16,79.74,Pass
4,16,Female,74.79,2.84,88.21,61.54,2.63,2.29,1,7.83,0.49,1,28,50.66,Pass
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,17,Male,78.73,NaN,89.67,60.97,1.16,0.68,7,7.00,4.29,1,5,66.46,Pass
99996,14,Female,75.70,2.98,83.87,94.38,8.59,6.63,12,6.73,3.60,1,24,96.54,Pass
99997,16,Female,84.69,3.27,89.57,71.28,2.45,2.20,5,7.20,1.62,1,24,74.66,Pass
99998,15,Female,74.70,3.06,92.61,69.26,4.77,3.92,7,8.37,2.21,1,28,85.85,Pass


## 5. Inspect the Selected Dataset

In [66]:

df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   age                         100000 non-null  int64  
 1   gender                      100000 non-null  object 
 2   previous_exam_score         100000 non-null  float64
 3   previous_gpa                92207 non-null   float64
 4   attendance_percentage       90137 non-null   float64
 5   assignment_completion_rate  100000 non-null  float64
 6   study_hours_per_day         100000 non-null  float64
 7   self_study_hours            100000 non-null  float64
 8   practice_tests_completed    100000 non-null  int64  
 9   sleep_hours                 100000 non-null  float64
 10  physical_activity_hours     100000 non-null  float64
 11  internet_access             100000 non-null  int64  
 12  exam_preparation_days       100000 non-null  int64  
 13  exam_score     

## 6. Train-Test Split

In [67]:
X = df2.drop(columns=['pass_status'])
y = df2['pass_status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


## 7. Filling Random Values

In [68]:
X_train_random = X_train.copy()
X_test_random = X_test.copy()


X_train_random.loc[X_train_random['previous_gpa'].isnull(),'previous_gpa'] = (
    X_train_random['previous_gpa'].dropna().sample(X_train_random['previous_gpa'].isnull().sum(),random_state=42).values
)

X_test_random.loc[
    X_test_random['previous_gpa'].isnull(),
    'previous_gpa'
] = (
    X_train_random['previous_gpa']
    .dropna()
    .sample(
        X_test_random['previous_gpa'].isnull().sum(),
        random_state=42
    )
    .values
)

X_train_random.loc[
    X_train_random['attendance_percentage'].isnull(),
    'attendance_percentage'
] = (
    X_train_random['attendance_percentage']
    .dropna()
    .sample(
        X_train_random['attendance_percentage'].isnull().sum(),
        random_state=42
    )
    .values
)

X_test_random.loc[
    X_test_random['attendance_percentage'].isnull(),
    'attendance_percentage'
] = (
    X_train_random['attendance_percentage']
    .dropna()
    .sample(
        X_test_random['attendance_percentage'].isnull().sum(),
        random_state=42
    )
    .values
)
print(X_train_random[['previous_gpa', 'attendance_percentage']].isnull().sum())
print(X_test_random[['previous_gpa', 'attendance_percentage']].isnull().sum())

previous_gpa             0
attendance_percentage    0
dtype: int64
previous_gpa             0
attendance_percentage    0
dtype: int64


## 8. Handling Missing Values

1. Mean Imputation
2. Median Imputation
3. Most Frequent Value Imputation
4. K-Nearest Neighbors (KNN) Imputation
5. MICE (Multiple Imputation by Chained Equations)
6. Row Removal
7. Random Imputation

In [69]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


transformer1 = ColumnTransformer(transformers=[
('tnf1',SimpleImputer(strategy='mean'),['previous_gpa','attendance_percentage']),
('tnf2',OneHotEncoder(sparse_output=False,drop='first'),['gender'])
] ,remainder='passthrough')

transformer2 = ColumnTransformer(transformers=[
('tnf1',SimpleImputer(strategy='median'),['previous_gpa','attendance_percentage']),
('tnf2',OneHotEncoder(sparse_output=False,drop='first'),['gender'])
] ,remainder='passthrough')

transformer3 = ColumnTransformer(transformers=[
('tnf1',SimpleImputer(strategy='most_frequent'),['previous_gpa','attendance_percentage']),
('tnf2',OneHotEncoder(sparse_output=False,drop='first'),['gender'])
] ,remainder='passthrough')

transformer4 = ColumnTransformer(transformers=[
('tnf1',KNNImputer(n_neighbors=5),['previous_gpa','attendance_percentage']),
('tnf2',OneHotEncoder(sparse_output=False,drop='first'),['gender'])
] ,remainder='passthrough')

transformer5 = ColumnTransformer(transformers=[
    ('tnf1', IterativeImputer(max_iter=10,random_state=42), ['previous_gpa', 'attendance_percentage']),
    ('tnf2', OneHotEncoder(sparse_output=False,drop='first'), ['gender'])
], remainder='passthrough')

transformer6 = ColumnTransformer(transformers=[
    ('tnf2', OneHotEncoder(sparse_output=False,drop='first'), ['gender'])
], remainder='passthrough')

transformer7 = ColumnTransformer(transformers=[
        ('tnf1', 'passthrough',['previous_gpa', 'attendance_percentage']),
        ('tnf2', OneHotEncoder(sparse_output=False,drop='first'), ['gender'])
    ], remainder='passthrough')

## 9. Building Machine Learning Pipelines,
##Feature Scaling,
##Logistic Regression,
##Generate Predictions,
##Compare Model Performance

In [70]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pickle

model1 = Pipeline([
    ('preprocessor', transformer1),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
model2 = Pipeline([
    ('preprocessor', transformer2),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
model3 = Pipeline([
    ('preprocessor', transformer3),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
model4 = Pipeline([
    ('preprocessor', transformer4),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
model5 = Pipeline([
    ('preprocessor', transformer5),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
model6 = Pipeline([
    ('preprocessor', transformer6),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
model7 = Pipeline([
    ('preprocessor', transformer7),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])




model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)
model4.fit(X_train, y_train)
model5.fit(X_train, y_train)

X_train_drop = X_train.dropna()
y_train_drop = y_train.loc[X_train_drop.index]
X_test_drop = X_test.dropna()
y_test_drop = y_test.loc[X_test_drop.index]

model6.fit(X_train_drop, y_train_drop)
model7.fit(X_train_random, y_train)

y_pred1 = model1.predict(X_test)
y_pred2 = model2.predict(X_test)
y_pred3 = model3.predict(X_test)
y_pred4 = model4.predict(X_test)
y_pred5 = model5.predict(X_test)
y_pred6 = model6.predict(X_test_drop)
y_pred7 = model7.predict(X_test_random)

print("Accuracy by mean:", accuracy_score(y_test, y_pred1))
print("Accuracy by median:", accuracy_score(y_test, y_pred2))
print("Accuracy by most frequent:", accuracy_score(y_test, y_pred3))
print("Accuracy by KNN:", accuracy_score(y_test, y_pred4))
print("Accuracy by MICE:", accuracy_score(y_test, y_pred5))
print("Accuracy by Row Removal:", accuracy_score(y_test_drop, y_pred6))
print("Accuracy by Random Imputation:", accuracy_score(y_test, y_pred7))

Accuracy by mean: 0.9993
Accuracy by median: 0.9993
Accuracy by most frequent: 0.99935
Accuracy by KNN: 0.9993
Accuracy by MICE: 0.9993
Accuracy by Row Removal: 0.9993389820323297
Accuracy by Random Imputation: 0.99925


## Conclusion

In this project, we built a classification pipeline to predict student pass status using Logistic Regression.

We compared seven different techniques for handling missing values:

1. Mean Imputation
2. Median Imputation
3. Most Frequent Imputation
4. K-Nearest Neighbors (KNN) Imputation
5. MICE (Multiple Imputation by Chained Equations)
6. Row Removal
7. Random Imputation

The models used the same train-test split, preprocessing structure, feature scaling, and Logistic Regression algorithm. The main difference between the models was the technique used to handle missing values.

After evaluating the models on the test dataset, the results were compared using accuracy. The comparison helped determine whether different missing-value handling techniques had an effect on the final classification performance.

For this particular dataset and model configuration, the imputation techniques produced similar results, showing that the choice of missing-value handling method did not have a significant effect on the classification accuracy. Row removal was evaluated separately because it removes observations containing missing values, while the other techniques retain those observations by estimating their missing values.

Overall, this project demonstrates how different missing-value handling techniques can be incorporated into a machine learning pipeline and compared using a consistent Logistic Regression model.
